# Week 2: Brownian Motion I: From Random Walks to Continuous-Time Risk

**Project:** *Valuing Variable Annuity Guarantees Under Market Crashes and Regime Shifts*  
**Mentor:** Hao Quan  
**Notebook type:** all-in-one lesson, lab, exercises, and worked solutions

> **Driving question:** How can many small independent shocks become the continuous process used in finance?

## Learning goals

- Construct scaled random walks and Brownian paths.
- Check stationary, independent, normally distributed increments.
- Verify mean, variance, covariance, and Brownian scaling empirically.
- Explain why Brownian paths are continuous but not smooth.
- Use quadratic variation as a computational signature of Brownian motion.

## How to use this notebook

1. Complete the assigned reading before the weekly meeting.
2. Read the explanation cells and predict each result before running the code.
3. Run the notebook from top to bottom. Every random experiment uses a fixed seed.
4. Attempt the exercises before opening the worked-solution section.
5. Write a 150–250 word interpretation of the main result in your own words.

This notebook is educational. It simplifies real contracts and is not an insurance
quotation, investment recommendation, or complete actuarial valuation.

## Assigned reading

- [MIT OCW Lecture 17: Stochastic Processes II](https://ocw.mit.edu/courses/18-s096-topics-in-mathematics-with-applications-in-finance-fall-2013/3b97c6b0c282dd9dc024c4c7ffe3fba8_MIT18_S096F13_lecnote17.pdf) — read all 7 pages, emphasizing Sections 12–13 and the random-walk limit


In [ ]:
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
np.set_printoptions(precision=4, suppress=True)

SEED = 20260730
rng = np.random.default_rng(SEED)
print(f"Reproducible random seed: {SEED}")


## 1. Definition used in this project

A standard Brownian motion $(W_t)_{t\ge0}$ satisfies:

1. $W_0=0$.
2. $W_t-W_s\sim N(0,t-s)$ for $0\le s<t$.
3. Increments over non-overlapping intervals are independent.
4. Sample paths are continuous.

Consequences include

$$E[W_t]=0,\qquad \operatorname{Var}(W_t)=t,$$

and

$$\operatorname{Cov}(W_s,W_t)=\min(s,t).$$

Simulation uses

$$\Delta W_k=\sqrt{\Delta t}Z_k,\qquad Z_k\sim N(0,1).$$


## 2. Start with a scaled symmetric random walk

For $n$ steps on $[0,1]$, take increments $\pm 1/\sqrt{n}$. The final
variance stays near one even as the grid becomes finer.


In [ ]:
def scaled_random_walk(n_steps, n_paths, generator):
    dt = 1 / n_steps
    signs = generator.choice([-1.0, 1.0], size=(n_paths, n_steps))
    increments = np.sqrt(dt) * signs
    paths = np.column_stack([np.zeros(n_paths), np.cumsum(increments, axis=1)])
    times = np.linspace(0, 1, n_steps + 1)
    return times, paths


fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, n_steps in zip(axes, [16, 64, 256]):
    local_rng = np.random.default_rng(SEED + n_steps)
    t, paths = scaled_random_walk(n_steps, 8, local_rng)
    ax.plot(t, paths.T, linewidth=1)
    ax.set(title=f"{n_steps} steps", xlabel="Time")
axes[0].set_ylabel("Scaled random walk")
fig.suptitle("Finer scaled random walks")
plt.tight_layout()
plt.show()


## 3. Direct Brownian-motion simulator


In [ ]:
def simulate_brownian(T=1.0, n_steps=252, n_paths=1, seed=SEED):
    if T <= 0 or n_steps < 1 or n_paths < 1:
        raise ValueError("T, n_steps, and n_paths must be positive")
    generator = np.random.default_rng(seed)
    dt = T / n_steps
    dW = np.sqrt(dt) * generator.standard_normal((n_paths, n_steps))
    W = np.column_stack([np.zeros(n_paths), np.cumsum(dW, axis=1)])
    times = np.linspace(0, T, n_steps + 1)
    return times, W, dW


times, W, dW = simulate_brownian(T=2, n_steps=504, n_paths=12)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(times, W.T, linewidth=1)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Twelve Brownian paths", xlabel="Time", ylabel="$W_t$")
plt.show()

assert np.allclose(W[:, 0], 0)
assert W.shape == (12, 505)


## 4. Moment and distribution checks

A plot can look convincing while code is wrong. We therefore compare simulated
moments with theoretical targets and use explicit tolerances.


In [ ]:
n_paths = 80_000
check_times, check_W, _ = simulate_brownian(
    T=2.0, n_steps=8, n_paths=n_paths, seed=SEED + 10
)
indices = [2, 4, 8]
rows = []
for idx in indices:
    sample = check_W[:, idx]
    t = check_times[idx]
    rows.append([
        t,
        sample.mean(),
        0.0,
        sample.var(ddof=1),
        t,
    ])

moment_check = pd.DataFrame(
    rows,
    columns=["time", "sample_mean", "target_mean", "sample_variance", "target_variance"],
)
print(moment_check.to_string(index=False))

assert abs(moment_check["sample_mean"]).max() < 0.02
assert abs(moment_check["sample_variance"] - moment_check["target_variance"]).max() < 0.04

terminal = check_W[:, -1]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(terminal, bins=70, density=True, alpha=0.65, label="Simulated $W_2$")
x = np.linspace(-5, 5, 500)
ax.plot(x, norm.pdf(x, loc=0, scale=np.sqrt(2)), color="black", label="$N(0,2)$ density")
ax.set(title="Terminal Brownian distribution", xlabel="$W_2$", ylabel="Density")
ax.legend()
plt.show()


## 5. Covariance: $\operatorname{Cov}(W_s,W_t)=\min(s,t)$


In [ ]:
target_times = np.array([0.25, 0.75, 1.25, 2.00])
target_idx = [int(round(t / 2.0 * 8)) for t in target_times]
sample_matrix = check_W[:, target_idx]
sample_cov = np.cov(sample_matrix, rowvar=False)
theory_cov = np.minimum.outer(target_times, target_times)

print("Sample covariance:")
print(pd.DataFrame(sample_cov, index=target_times, columns=target_times).round(3))
print("\nTheoretical covariance:")
print(pd.DataFrame(theory_cov, index=target_times, columns=target_times).round(3))

assert np.max(np.abs(sample_cov - theory_cov)) < 0.05


## 6. Independent increments

Zero correlation is weaker than independence in general. Here the increments
are jointly normal, so near-zero empirical correlations are a useful check.


In [ ]:
inc_a = check_W[:, 2] - check_W[:, 0]  # [0, 0.5]
inc_b = check_W[:, 6] - check_W[:, 4]  # [1.0, 1.5]
inc_overlap = check_W[:, 4] - check_W[:, 1]  # overlaps inc_a

corr_nonoverlap = np.corrcoef(inc_a, inc_b)[0, 1]
corr_overlap = np.corrcoef(inc_a, inc_overlap)[0, 1]
print(f"Non-overlapping increment correlation: {corr_nonoverlap:.4f}")
print(f"Overlapping increment correlation:     {corr_overlap:.4f}")
assert abs(corr_nonoverlap) < 0.02


## 7. Brownian scaling

For any $c>0$, the process $(W_{ct}/\sqrt{c})$ has the same distribution as
$(W_t)$. We check this at a fixed time with two independent samples.


In [ ]:
scale_rng = np.random.default_rng(SEED + 20)
n = 100_000
W_1 = scale_rng.normal(0, 1, n)
W_4_scaled = scale_rng.normal(0, np.sqrt(4), n) / np.sqrt(4)

scaling_check = pd.DataFrame({
    "sample": ["W_1", "W_4 / sqrt(4)"],
    "mean": [W_1.mean(), W_4_scaled.mean()],
    "variance": [W_1.var(ddof=1), W_4_scaled.var(ddof=1)],
    "q05": [np.quantile(W_1, 0.05), np.quantile(W_4_scaled, 0.05)],
    "q95": [np.quantile(W_1, 0.95), np.quantile(W_4_scaled, 0.95)],
})
print(scaling_check.to_string(index=False))
assert np.max(np.abs(scaling_check["variance"] - 1)) < 0.02


## 8. Quadratic variation: continuous does not mean smooth

For a smooth function, the sum of squared increments tends to zero as the grid
is refined. For Brownian motion,

$$\sum_k (W_{t_{k+1}}-W_{t_k})^2 \longrightarrow T.$$

This nonzero quadratic variation is why ordinary calculus must be modified.


In [ ]:
def mean_quadratic_variation(T, n_steps, n_paths=4_000, seed=SEED):
    _, _, increments = simulate_brownian(T, n_steps, n_paths, seed)
    qv = np.sum(increments**2, axis=1)
    return qv.mean(), qv.std(ddof=1)


qv_rows = []
for n_steps in [16, 64, 256, 1024]:
    mean_qv, sd_qv = mean_quadratic_variation(
        T=1.0, n_steps=n_steps, seed=SEED + n_steps
    )
    qv_rows.append([n_steps, mean_qv, sd_qv])

qv_table = pd.DataFrame(qv_rows, columns=["steps", "mean_QV", "sd_QV"])
print(qv_table.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(qv_table["steps"], qv_table["mean_QV"], marker="o")
ax.axhline(1, color="black", linestyle="--", label="Theoretical limit $T=1$")
ax.set(title="Realized quadratic variation", xlabel="Number of time steps", ylabel="Mean QV")
ax.legend()
plt.show()

assert abs(qv_table.iloc[-1]["mean_QV"] - 1) < 0.02
assert qv_table.iloc[-1]["sd_QV"] < qv_table.iloc[0]["sd_QV"]


## 9. Optional first-passage experiment

The first time a process crosses a level is a path-dependent random variable.
This idea later matters for barriers, withdrawals, and ruin—even though the
core GMAB payoff depends only on maturity.


In [ ]:
fp_times, fp_W, _ = simulate_brownian(T=1, n_steps=500, n_paths=20_000, seed=SEED + 30)
level = 1.0
crossed = fp_W >= level
ever_crossed = crossed.any(axis=1)
first_index = np.where(ever_crossed, crossed.argmax(axis=1), -1)
first_times = fp_times[first_index[ever_crossed]]

print(f"Estimated P(max W_t >= {level} before T=1): {ever_crossed.mean():.4f}")
print(f"Reflection-principle target:               {2 * (1 - norm.cdf(level)):.4f}")
print(f"Median first passage time, conditional on crossing: {np.median(first_times):.4f}")
assert abs(ever_crossed.mean() - 2 * (1 - norm.cdf(level))) < 0.02


## 10. Exercises

1. Simulate $W_3$ and verify its mean and variance.
2. Estimate $\operatorname{Cov}(W_{0.4},W_{1.6})$.
3. Show numerically that the standard deviation of an increment of length
   $\Delta t$ is proportional to $\sqrt{\Delta t}$.
4. In words, reconcile “continuous paths” with “nowhere differentiable.”


## 11. Worked solutions


In [ ]:
# Exercises 1 and 2
sol_t, sol_W, _ = simulate_brownian(T=2.0, n_steps=10, n_paths=100_000, seed=SEED + 40)
W_04 = sol_W[:, 2]
W_16 = sol_W[:, 8]

separate_rng = np.random.default_rng(SEED + 41)
W_3 = np.sqrt(3) * separate_rng.standard_normal(100_000)
print(f"W_3 mean={W_3.mean():.4f}, variance={W_3.var(ddof=1):.4f}")
print(f"Cov(W_0.4, W_1.6)={np.cov(W_04, W_16, ddof=1)[0,1]:.4f}; target=0.4")

# Exercise 3
dt_values = np.array([1/12, 1/52, 1/252])
sd_empirical = []
scale_rng = np.random.default_rng(SEED + 42)
for dt in dt_values:
    increments = np.sqrt(dt) * scale_rng.standard_normal(100_000)
    sd_empirical.append(increments.std(ddof=1))
sd_check = pd.DataFrame({
    "dt": dt_values,
    "empirical_sd": sd_empirical,
    "sqrt_dt": np.sqrt(dt_values),
})
print(sd_check.to_string(index=False))

assert abs(W_3.var(ddof=1) - 3) < 0.05
assert abs(np.cov(W_04, W_16, ddof=1)[0,1] - 0.4) < 0.02


### Interpretation checkpoint

Brownian paths are continuous because there are no jumps, but their increments
remain rough at every scale. The squared increments accumulate to a nonzero
limit, so an ordinary derivative does not exist. In Week 3, that accumulated
quadratic variation produces the $-\sigma^2/2$ correction in geometric
Brownian motion.

### Weekly submission

- Completed notebook
- A 300-word explanation of continuous-but-not-smooth paths
- Three numerical Brownian-property checks with targets and tolerances
